# Faruq-v3 — frozen-D0 multilevel residual screening

FRM1 mempertahankan seluruh D0 dan melatih hanya residual classifier serta confidence gate selama 10 epoch. Static preservation gate dijalankan sebelum dataset training. Test tetap tidak tersedia.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
BASELINE = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-frozen-residual-v1'
STATIC_AUDIT = OUTPUT_ROOT / 'static_audit.json'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('D0        :', D0_CHECKPOINT)
print('OUTPUT    :', OUTPUT_ROOT)
last = OUTPUT_ROOT / 'FRM1_seed42/weights/last.pt'
best = OUTPUT_ROOT / 'FRM1_seed42/weights/best.pt'
print('FRM1      :', 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))

In [ ]:
from coffee_detector.frozen_residual.audit import static_frozen_residual_audit
static = static_frozen_residual_audit(
    REPO / 'configs/coffee_fg/models/yolo26n-p3.yaml',
    D0_CHECKPOINT, STATIC_AUDIT, nc=21, image_size=128, topk=32,
)
print('STATIC GATES:', static['gates'])
print('IDENTITY DIFF:', static['zero_output_max_abs_diff'])
print('TRAINABLE:', static['parameter_counts'])
print('DECISION:', static['decision'])
assert static['decision'] == 'PASS', 'STOP: D0-preservation static gate gagal.'

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_frozen_residual',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--baseline-summary', str(BASELINE),
    '--d0-checkpoint', str(D0_CHECKPOINT),
    '--static-audit', str(STATIC_AUDIT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT / 'val_reports/frozen_residual_seed42_decision.json'
assert SUMMARY.is_file(), f'FRM1 belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = [{'model': name, **metrics} for name, metrics in result['results'].items()]
display(pd.DataFrame(rows).style.format({name: '{:.2%}' for name in ('macro_map50_95', 'bottom3_class_map50_95', 'worst_class_map50_95')}))
print('DELTAS  :', result['deltas'])
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
print('Kirim tabel dan keputusan. Jangan membuka test atau menjalankan seed lain.')